In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
from kaggle_secrets import UserSecretsClient
api_key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")

In [ ]:
!pip install -q roboflow torchmetrics torchsummary
import torch
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())


In [ ]:
from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow

api_key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
rf = Roboflow(api_key=api_key)
project = rf.workspace("vivek-gangurde").project("kabaddi-player-detection")
dataset = project.version(1).download("yolov8")
print("Saved at:", dataset.location)

class names and image count

In [ ]:
import yaml
from pathlib import Path

root = Path(dataset.location)
with open(root / "data.yaml") as f:
    cfg = yaml.safe_load(f)

names = cfg["names"]
class_names = [names[k] for k in sorted(names)] if isinstance(names, dict) else list(names)
print("Classes:", class_names)

for split in ["train", "valid", "test"]:
    p = root / split / "images"
    print(split, "images:", len(list(p.glob("*"))) if p.exists() else "missing")

boxes per class****

In [ ]:
from collections import Counter

for split in ["train", "valid", "test"]:
    c = Counter()
    lab_dir = root / split / "labels"
    if lab_dir.exists():
        for lf in lab_dir.glob("*.txt"):
            for line in lf.read_text().splitlines():
                if line.strip():
                    c[int(line.split()[0])] += 1
    print(split, {class_names[k]: v for k, v in sorted(c.items())})

****crop every box into class folders****

In [ ]:
from PIL import Image

out_root = Path("/kaggle/working/crops")
split_map = {"train": "train", "valid": "val", "test": "test"}
MIN_SIZE = 20  # skip tiny boxes

for src, dst in split_map.items():
    for cls in class_names:
        (out_root / dst / cls).mkdir(parents=True, exist_ok=True)
    img_dir = root / src / "images"
    if not img_dir.exists():
        continue
    for img_path in img_dir.glob("*"):
        lab = root / src / "labels" / (img_path.stem + ".txt")
        if not lab.exists():
            continue
        img = Image.open(img_path).convert("RGB")
        W, H = img.size
        for i, line in enumerate(lab.read_text().splitlines()):
            parts = line.split()
            if len(parts) < 5:
                continue
            c = int(parts[0])
            x, y, w, h = map(float, parts[1:5])
            x1, y1 = max(0, int((x - w / 2) * W)), max(0, int((y - h / 2) * H))
            x2, y2 = min(W, int((x + w / 2) * W)), min(H, int((y + h / 2) * H))
            if (x2 - x1) < MIN_SIZE or (y2 - y1) < MIN_SIZE:
                continue
            img.crop((x1, y1, x2, y2)).save(out_root / dst / class_names[c] / f"{img_path.stem}_{i}.jpg")

for d in ["train", "val", "test"]:
    for cls in class_names:
        print(d, cls, len(list((out_root / d / cls).glob("*"))))

# import

In [ ]:
import os
import csv, copy, time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchmetrics.classification import MulticlassPrecision, MulticlassRecall, MulticlassF1Score

# transforms

In [ ]:
norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(), norm
])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(), norm
])

# load the data

In [ ]:
train_set = ImageFolder(str(out_root / "train"), train_tf)
val_set   = ImageFolder(str(out_root / "val"),   eval_tf)
test_set  = ImageFolder(str(out_root / "test"),  eval_tf)

num_classes = len(train_set.classes)
print("Classes:", train_set.classes, "| count:", num_classes)
print("Class indices:", train_set.class_to_idx)
print("Sizes -> train:", len(train_set), "val:", len(val_set), "test:", len(test_set))

train_loader = DataLoader(train_set, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_set,   batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=32, shuffle=False, num_workers=2)

# build the model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

for param in model.parameters():      # freeze the pretrained backbone
    param.requires_grad = False

model.fc = nn.Sequential(             # new trainable head
    nn.Linear(model.fc.in_features, 128),
    nn.ReLU(),
    nn.Linear(128, num_classes)
)
model = model.to(device)
print(next(model.parameters()).device)

# loss and optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# the training function


In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs, device, num_classes):
    since = time.time()
    best_val_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())
    best_epoch = 0

    # Create the CSV log
    csv_filename = 'training_log.csv'
    with open(csv_filename, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['Epoch', 'Train Loss', 'Train Acc', 'Train Precision', 'Train Recall', 'Train F1',
                         'Val Loss', 'Val Acc', 'Val Precision', 'Val Recall', 'Val F1'])

    # Metrics (macro = average over classes)
    train_precision = MulticlassPrecision(num_classes=num_classes, average='macro').to(device)
    train_recall = MulticlassRecall(num_classes=num_classes, average='macro').to(device)
    train_f1 = MulticlassF1Score(num_classes=num_classes, average='macro').to(device)

    val_precision = MulticlassPrecision(num_classes=num_classes, average='macro').to(device)
    val_recall = MulticlassRecall(num_classes=num_classes, average='macro').to(device)
    val_f1 = MulticlassF1Score(num_classes=num_classes, average='macro').to(device)

    for epoch in range(epochs):
        # ---------- Training ----------
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        all_train_preds = []
        all_train_labels = []

        for inputs, labels in train_loader:
            inputs = inputs.to(device).float()
            labels = labels.to(device).long()

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)   # FIXED
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_train_preds.append(predicted)
            all_train_labels.append(labels)

        avg_train_loss = running_loss / len(train_loader.dataset)
        train_accuracy = 100 * correct / total
        all_train_preds = torch.cat(all_train_preds)
        all_train_labels = torch.cat(all_train_labels)

        train_precision_score = train_precision(all_train_preds, all_train_labels).item()
        train_recall_score = train_recall(all_train_preds, all_train_labels).item()
        train_f1_score = train_f1(all_train_preds, all_train_labels).item()

        print(f"Epoch [{epoch+1}/{epochs}], Training Loss: {avg_train_loss:.4f}, "
              f"Training Accuracy: {train_accuracy:.2f}%, "
              f"Train Precision: {train_precision_score:.4f}, Train Recall: {train_recall_score:.4f}, Train F1: {train_f1_score:.4f}")

        # ---------- Validation ----------
        model.eval()
        val_loss = 0.0
        val_corrects = 0
        all_val_preds = []
        all_val_labels = []

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                val_corrects += (predicted == labels).sum().item()
                all_val_preds.append(predicted)
                all_val_labels.append(labels)

        avg_val_loss = val_loss / len(val_loader.dataset)
        val_accuracy = 100 * val_corrects / len(val_loader.dataset)
        all_val_preds = torch.cat(all_val_preds)
        all_val_labels = torch.cat(all_val_labels)

        val_precision_score = val_precision(all_val_preds, all_val_labels).item()
        val_recall_score = val_recall(all_val_preds, all_val_labels).item()
        val_f1_score = val_f1(all_val_preds, all_val_labels).item()

        print(f"Epoch [{epoch+1}/{epochs}], Val Loss: {avg_val_loss:.4f}, Val Acc: {val_accuracy:.2f}% | "
              f"Precision: {val_precision_score:.4f}, Recall: {val_recall_score:.4f}, F1: {val_f1_score:.4f}")

        # Write this epoch to the CSV
        with open(csv_filename, mode='a', newline='') as file:
            writer = csv.writer(file)
            writer.writerow([epoch+1, avg_train_loss, train_accuracy, train_precision_score, train_recall_score, train_f1_score,
                             avg_val_loss, val_accuracy, val_precision_score, val_recall_score, val_f1_score])

        # Keep the best model
        if val_accuracy > best_val_acc:
            best_val_acc = val_accuracy
            best_model_wts = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1

    elapsed_time = time.time() - since
    print(f"Training complete in {elapsed_time // 60:.0f}m {elapsed_time % 60:.0f}s")

    torch.save(best_model_wts, f"best_model_epoch_{best_epoch}.pth")
    print(f"\nBest model saved at epoch {best_epoch} with {best_val_acc:.2f}% accuracy")

    model.load_state_dict(best_model_wts)
    return model

# train

In [ ]:
model = train_model(model, train_loader, val_loader, criterion, optimizer,
                    epochs=10, device=device, num_classes=num_classes)

# test the best model and save the confusion matrix

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
preds, truth = [], []
with torch.no_grad():
    for x, y in test_loader:
        out = model(x.to(device))
        preds += out.argmax(1).cpu().tolist()
        truth += y.tolist()

print(classification_report(truth, preds, target_names=test_set.classes, digits=4))

cm = confusion_matrix(truth, preds)
cm_df = pd.DataFrame(cm, index=test_set.classes, columns=test_set.classes)
cm_df.to_csv("confusion_matrix.csv")
print(cm_df)